In [4]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib





#Loading and cleaning s&p 500

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")


valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns)
)
close_df, high_df, low_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
)


[*****                 11%                       ]  53 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[************          26%                       ]  130 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************46%                       ]  230 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['HONA', 'FDXF', 'VYLR']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


In [10]:
#Awesome oscilator for s&p500


median_price = (high_df + low_df) / 2


awesome_dict = {}

for ticker in median_price.columns:
    prices = median_price[ticker].to_numpy(dtype=np.float64)

    sma5 = talib.SMA(prices, timeperiod=7)
    sma34 = talib.SMA(prices, timeperiod=56)

    awesome_dict[ticker] = sma5 - sma34

awesome = pd.DataFrame(
    awesome_dict,
    index=median_price.index
)


entries = (awesome < 0) & (awesome.shift(1) > 0)
exits = (awesome > 0) & (awesome.shift(1) < 0)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=10_000.0,
    size_type="value",
    cash_sharing = True,
    group_by= True,
    freq="1D",
    fees=0.001
)

# 8. View results
print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               444195.192011
Total Return [%]                           344.195192
Benchmark Return [%]                       524.559296
Max Gross Exposure [%]                          100.0
Total Fees Paid                          48651.901522
Max Drawdown [%]                            43.860074
Max Drawdown Duration               423 days 00:00:00
Total Trades                                     3553
Total Closed Trades                              3484
Total Open Trades                                  69
Open Trade PnL                          -25389.638032
Win Rate [%]                                73.622273
Best Trade [%]                              57.024263
Worst Trade [%]                            -71.199649
Avg Winning Trade [%]       

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



In [ ]:
#Awesome oscilator for s&p500, diff moving average (7, 56) 
# WORKS BETTER THAN (5,34)


median_price = (high_df + low_df) / 2


awesome_dict = {}

for ticker in median_price.columns:
    prices = median_price[ticker].to_numpy(dtype=np.float64)

    sma5 = talib.SMA(prices, timeperiod=7)
    sma34 = talib.SMA(prices, timeperiod=56)

    awesome_dict[ticker] = sma5 - sma34

awesome = pd.DataFrame(
    awesome_dict,
    index=median_price.index
)


entries = (awesome < 0) & (awesome.shift(1) > 0)
exits = (awesome > 0) & (awesome.shift(1) < 0)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=10_000.0,
    size_type="value",
    cash_sharing = True,
    group_by= True,
    freq="1D",
    fees=0.001
)

# 8. View results
print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               444195.192011
Total Return [%]                           344.195192
Benchmark Return [%]                       524.559296
Max Gross Exposure [%]                          100.0
Total Fees Paid                          48651.901522
Max Drawdown [%]                            43.860074
Max Drawdown Duration               423 days 00:00:00
Total Trades                                     3553
Total Closed Trades                              3484
Total Open Trades                                  69
Open Trade PnL                          -25389.638032
Win Rate [%]                                73.622273
Best Trade [%]                              57.024263
Worst Trade [%]                            -71.199649
Avg Winning Trade [%]       

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

/opt/anaconda3/lib/python3.13/site-packages/vectorbt/generic/plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data

